# 01 — Smoke test: Full CoT vs. DEER (τ = 0.95)

Runs DeepSeek-R1-Distill-Qwen-1.5B on the first `N` questions of GSM8K and MATH-500 with
1. **Full CoT**: plain greedy generation (`scripts/run_full_cot.py`)
2. **DEER τ=0.95**: the official DEER loop plus our logging (`third_party/deer/vllm_deer_logged.py`)
3. *(optional)* **Never-exit DEER**: τ > 1, which logs a trial answer at every checkpoint. This is the trajectory data for the threshold sweep and the controller.

Then it scores everything and draws the accuracy vs. tokens plot.

**Runtime:** Colab → *Runtime → Change runtime type → T4 GPU*. A T4 has no bf16 support, so we use `float16`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Clones the public team repo and moves into this folder (`Vishnu-evaluation/`).

In [ ]:
REPO = "https://github.com/PeramVishnuSree/Adaptive-Early-Exit-for-Efficient-Reasoning-Reproducing-and-Extending-DEER.git"
!git clone -q $REPO /content/repo || (cd /content/repo && git pull -q)
%cd /content/repo/Vishnu-evaluation
!git log --oneline -1

## 2. Install
Colab runs Python 3.13, so we use **vLLM 0.26.0**, the release built for Colab's torch 2.11.
(The official DEER code used `vllm<=0.6.1`, which can't be installed on Python 3.13.)
Exit logic and decoding settings are unchanged.

`latex2sympy2_extended` replaces DEER's `latex2sympy2==1.9.1`, whose pinned ANTLR version conflicts with sympy's.
Do **not** pin `sympy==1.12`: torch 2.11 needs `sympy>=1.13.3`.

In [ ]:
!pip install -q "vllm==0.26.0" nltk
!pip install -q "antlr4-python3-runtime==4.11.0" "latex2sympy2_extended[antlr4_11_0]" regex word2number

Check the install. The second line loads the model in a separate process (so this notebook doesn't hold GPU
memory) and generates a few tokens. Expect `vllm 0.26.0 | torch 2.11.0+... | cuda True`, then some text.
The first load takes a few minutes (model download + compile).

In [ ]:
!python -c "import vllm, torch, sympy; print('vllm', vllm.__version__, '| torch', torch.__version__, '| sympy', sympy.__version__, '| cuda', torch.cuda.is_available())"
!python -c "from vllm import LLM, SamplingParams; llm = LLM('deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', dtype='float16', max_model_len=4096, gpu_memory_utilization=0.8); print(repr(llm.generate(['What is 2+3?'], SamplingParams(max_tokens=20, temperature=0))[0].outputs[0].text))"

**Only if the check above fails with a CUDA error** (e.g. `undefined symbol` or `libcudart.so.12`): vLLM's wheels
are built for CUDA 12.9, but Colab's torch is the CUDA 13 build. Set `FIX_CUDA = True`, run the cell, then
*Runtime → Restart session* and re-run the check cell.

In [ ]:
FIX_CUDA = False
if FIX_CUDA:
    !pip install -q "torch==2.11.0" "torchvision==0.26.0" "torchaudio==2.11.0" --index-url https://download.pytorch.org/whl/cu129

## 3. Settings
Keep `N` small first (e.g. 10) to check everything runs, then raise it to 50–100.

In [ ]:
import os
%cd /content/repo/Vishnu-evaluation
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
N = 50                  # questions per dataset (prefix of the test set)
MAX_TOKENS = 16000      # DEER paper setting (total generation budget)
DATASETS = ["gsm8k", "math"]
DTYPE = "float16"       # T4/V100. Use bfloat16 on A100/H100.
RUN_NEVER_EXIT = False  # set True for the trajectory-logging run (slowest of the three)

OUT = "outputs/smoke"
os.makedirs(OUT, exist_ok=True)

## 4. Full CoT baseline

In [ ]:
for ds in DATASETS:
    !python scripts/run_full_cot.py --model $MODEL --dataset $ds --limit $N \
        --max_generated_tokens $MAX_TOKENS --dtype $DTYPE --output_file $OUT/fullcot_{ds}.jsonl

## 5. DEER, τ = 0.95
Official settings for DeepSeek-R1-Distill models from the DEER README: `--think_ratio 0.6 --policy avg1`.

In [ ]:
for ds in DATASETS:
    !python third_party/deer/vllm_deer_logged.py --model_name_or_path $MODEL \
        --dataset_dir data --dataset $ds --limit $N --threshold 0.95 \
        --max_generated_tokens $MAX_TOKENS --think_ratio 0.6 --policy avg1 \
        --dtype $DTYPE --gpu-memory-utilization 0.9 \
        --output_file $OUT/deer_t0.95_{ds}.jsonl 2>&1 | grep -v "Early exit threshold not reached" | tail -5

## 6. (Optional) Never-exit run: logs every checkpoint
τ = 1.1 can never be reached, so the model thinks to the end while a trial answer + confidence is
recorded at every "Wait". `--max_judge_steps 50` raises DEER's default cap of 10 checkpoints.

In [ ]:
if RUN_NEVER_EXIT:
    for ds in DATASETS:
        !python third_party/deer/vllm_deer_logged.py --model_name_or_path $MODEL \
            --dataset_dir data --dataset $ds --limit $N --threshold 1.1 --max_judge_steps 50 \
            --max_generated_tokens $MAX_TOKENS --think_ratio 0.6 --policy avg1 \
            --dtype $DTYPE --gpu-memory-utilization 0.9 \
            --output_file $OUT/never_{ds}.jsonl 2>&1 | grep -v "Early exit threshold not reached" | tail -5

## 7. Score and compare
Needs the evaluation code (commit 2). The first cell pulls the latest version of the repo.
You can also skip this, download the zip in step 8, and score on a laptop.

In [ ]:
!git pull -q
import glob
for f in sorted(glob.glob(f"{OUT}/*.jsonl")):
    if f.endswith(".scored.jsonl"):
        continue
    ds = "gsm8k" if f.endswith("_gsm8k.jsonl") else "math"
    !python scripts/score_run.py --input $f --dataset $ds

In [ ]:
for ds in DATASETS:
    sweep = f"--sweep {OUT}/never_{ds}.scored.jsonl" if os.path.exists(f"{OUT}/never_{ds}.scored.jsonl") else ""
    !python scripts/compare_runs.py --runs "Full CoT={OUT}/fullcot_{ds}.scored.jsonl" \
        "DEER t=0.95={OUT}/deer_t0.95_{ds}.scored.jsonl" --baseline "Full CoT" $sweep \
        --out_dir results/smoke_{ds} --title "{ds.upper()} first {N}, R1-Distill-1.5B"

In [ ]:
from IPython.display import Image, display
for ds in DATASETS:
    display(Image(f"results/smoke_{ds}/tradeoff.png", width=500))

## 8. Save outputs
Download the zip (or copy it to the team Google Drive folder). Raw outputs are gitignored. Commit only
`results/` (small tables and figures).

In [ ]:
!zip -qr smoke_outputs.zip outputs/smoke results
from google.colab import files
files.download("smoke_outputs.zip")